<- [18 - Einsteins Riddle](18_Einsteins_Riddle.ipynb) | [README Z3](README.md)

***

# 20 — Étendre le DSL : réécriture de prédicats et réécriture globale

> Notebook de la série **Z3 / SMT** (track C#, fork [Z3.Linq](../Z3.Linq/)).
> Capacité **jamais exercée dans les carnets de la série** — vérifié par grep des deux
> attributs sur les carnets 01-18 (zéro occurrence) ; couverte uniquement par les tests
> unitaires du fork (`Z3.Linq.Tests/RewriterTests.cs`), jamais montrée en notebook :
> les **points d'extension de réécriture** — `[TheoremPredicateRewriter]` et
> `[TheoremGlobalRewriter]` — qui permettent d'adapter le DSL à un vocabulaire métier
> **sans modifier la bibliothèque**.

## Le problème

Un DSL n'est jamais complet. Le traducteur de Z3.Linq sait traduire l'arithmétique, les
comparaisons, les tableaux, les `record`… mais pas *vos* méthodes. Que faire quand une contrainte
s'écrit naturellement `t => TousDifferents(t.A, t.B, t.C)` alors que `TousDifferents` n'a aucun
sens pour Z3 ? Trois issues :

1. inliner la contrainte à la main partout — répétition, perte du vocabulaire métier ;
2. forker la bibliothèque et ajouter un cas au traducteur — lourd, irreproductible ;
3. **brancher un rewriter** : laisser la bibliothèque appeler *votre* code au moment où elle
   rencontre la construction inconnue, et lui rendre une expression qu'elle sait traduire.

C'est le troisième chemin que ce carnet démontre, aux deux granularités offertes par le fork :
**au niveau d'un appel de méthode** (prédicat) et **au niveau de l'ensemble des contraintes**
(global).

## Ce que le carnet mesure, et pas seulement affirme

Chaque mécanisme est jugé par ce qu'il **change au verdict du solveur** :

- un prédicat métier refusé par le traducteur devient résoluble **grâce** au rewriter ;
- un rewriter **identité** est refusé par la bibliothèque (garde anti-boucle) ;
- une cible de réécriture **non traduisible** est refusée aussi — la contrainte de conception
  du mécanisme, mesurée et non supposée ;
- un rewriter **global** qui supprime une contrainte renverse un `UNSAT` en `SAT`, et
  `Explain()` rapporte la contrainte **injectée** par le rewriter — la cohérence
  diagnostic / solveur est montrée, pas affirmée.

## Ce que ce carnet n'est pas

Il ne ré-implémente pas le traducteur (organ-first : l'organe est dans le fork, le carnet
l'invoque), ne corrige pas la bibliothèque, et ne couvre ni `SolverKind` / `.Logic` / `.Tactics`
(carnet 19), ni les quantificateurs bornés `ForAll` / `Exists` (traités ailleurs dans la série).


## La surface publique : deux attributs, deux points d'appel

Le fork expose ces deux mécanismes par des **attributs** — c'est toute la surface publique :

| Attribut | Porte sur | Effet |
|---|---|---|
| `[TheoremGlobalRewriter(typeof(R))]` | le **type d'environnement** | `R` reçoit toutes les contraintes du théorème et renvoie la liste à asserter |
| `[TheoremPredicateRewriter(typeof(R))]` | une **méthode de prédicat** | `R` reçoit l'appel de méthode et renvoie l'appel à traduire à sa place |

Mesure de l'absence, rejouable, sur `origin/main` avant ce carnet :

```
git grep -o -E "ITheoremGlobalRewriter|ITheoremPredicateRewriter|TheoremGlobalRewriter|TheoremPredicateRewriter" \
    origin/main -- 'MyIA.AI.Notebooks/**/*.ipynb' | wc -l
0
```

Le fork implémente pourtant cette surface, fichiers et points d'appel citables : les interfaces et leurs attributs (`solutions/Z3.Linq/ITheoremGlobalRewriter.cs`, `ITheoremPredicateRewriter.cs`, `TheoremGlobalRewriterAttribute.cs`, `TheoremPredicateRewriterAttribute.cs`) et les deux points d'appel — `Theorem.cs:396-418` (`GetConstraintsToAssert`, partagé par le chemin de résolution et par `Explain`) et `ExpressionVisitor.cs:446-471` (`VisitCall`, avec la garde anti-boucle). C'est une capacité **publique, réelle et jamais exposée** en carnet avant celui-ci.


In [1]:
#r "../Z3.Linq/.deploy/Microsoft.Z3.dll"
#r "../Z3.Linq/.deploy/ExpressionUtils.dll"
#r "../Z3.Linq/.deploy/Z3.Linq.dll"
using Z3.Linq;
using Microsoft.Z3;
using System;
using System.Linq;
using System.Linq.Expressions;
using System.Collections.Generic;
using System.Text;

Console.OutputEncoding = Encoding.UTF8;
Console.WriteLine("Imports OK : Z3.Linq (.deploy du fork) + Microsoft.Z3 + arbres d'expression");
Console.WriteLine($"Point d'extension 1 : {typeof(ITheoremPredicateRewriter).FullName}");
Console.WriteLine($"Point d'extension 2 : {typeof(ITheoremGlobalRewriter).FullName}");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Imports OK : Z3.Linq (.deploy du fork) + Microsoft.Z3 + arbres d'expression


Point d'extension 1 : Z3.Linq.ITheoremPredicateRewriter


Point d'extension 2 : Z3.Linq.ITheoremGlobalRewriter


## 1. Le point de blocage, mesuré sans rewriter

Un triplet d'entiers `(A, B, C)` devant prendre trois valeurs **deux à deux distinctes**, positives,
de somme 6 : les solutions sont les permutations de `(1, 2, 3)`. En C# métier, la contrainte
s'écrit `TousDifferents(A, B, C)`.

Confrontons cette écriture au traducteur, sans aucune aide.


In [2]:
// Un predicat metier, volontairement SANS attribut de reecriture.
public static class PredicatsSansAttribut
{
    public static bool TousDifferents(int a, int b, int c) => a != b && b != c && a != c;
}

public class Triplet
{
    public int A { get; set; }
    public int B { get; set; }
    public int C { get; set; }
}

using (var ctx = new Z3Context())
{
    try
    {
        var theorem = from t in ctx.NewTheorem<Triplet>()
                      where PredicatsSansAttribut.TousDifferents(t.A, t.B, t.C)
                      where t.A > 0 && t.B > 0 && t.C > 0
                      where t.A + t.B + t.C == 6
                      select t;
        var sol = theorem.Solve();
        Console.WriteLine($"INATTENDU : solution trouvee -> {sol}");
    }
    catch (Exception ex)
    {
        Console.WriteLine($"REFUS attendu : {ex.GetType().Name}");
        Console.WriteLine($"  {ex.Message}");
    }
}


REFUS attendu : NotSupportedException


  Unknown method call:Boolean TousDifferents(Int32, Int32, Int32)



warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, 

### Lecture du résultat

Le traducteur ne devine pas qu'un appel de méthode signifie autre chose. Sa table de traduction
couvre des **nœuds d'expression** (arithmétique, comparaisons, conversions) et une liste close de
méthodes « magiques » ; tout le reste lève `NotSupportedException`.

C'est exactement le trou que comble le rewriter — et non un contournement par « inliner à la main
partout », qui ferait perdre le vocabulaire métier dans chaque contrainte.


## 2. Le contrat du rewriter de prédicat

Lu dans le fork (`ITheoremPredicateRewriter.cs`, `ExpressionVisitor.VisitCall`) :

| Règle | Portée |
|---|---|
| Attribut `[TheoremPredicateRewriter(typeof(...))]` posé sur **une méthode** | c'est l'**appel** de cette méthode qui déclenche la réécriture |
| Constructeur **public sans paramètre** du rewriter | la bibliothèque l'instancie par `Activator.CreateInstance` |
| `Rewrite(MethodCallExpression call)` rend un **`MethodCallExpression`** | pas n'importe quelle `Expression` : un *appel* |
| Le résultat doit **différer** de l'entrée | sinon `InvalidOperationException` immédiate — garde anti-boucle |

### La contrainte de conception, vérifiée à la source

`VisitCall` ne fait **pas** d'inlining de corps de méthode : après la réécriture, l'expression
rendue est re-visitée telle quelle, et si elle n'appartient pas à la table de traduction, elle est
refusée à son tour. **Un rewriter ne peut donc pas se contenter de « déplier » vers du C# arbitraire :
sa cible doit être une forme que le traducteur sait déjà traduire.** Dans le fork, ces formes sont
les opérateurs « magiques » de `Z3Methods` — `Distinct`, `Sum`, `ExactlyOne`, `AtMostOne`,
`AtLeastOne`, `Weighted*`, `ForAll`, `Exists` — et les accesseurs indexés.

C'est ce qui rend le mécanisme utile : on habille d'un vocabulaire métier des opérateurs que le
traducteur traite déjà, mais que personne n'a envie d'écrire à la main dans une contrainte.
La section 4 en fait la démonstration négative.


In [3]:
// Rewriters : chacun rend un appel que le traducteur sait deja traiter.
public class DeplieurMetier : ITheoremPredicateRewriter
{
    private static readonly System.Reflection.MethodInfo DistinctInt =
        typeof(Z3Methods).GetMethod(nameof(Z3Methods.Distinct))!.MakeGenericMethod(typeof(int));

    private static readonly System.Reflection.MethodInfo ExactlyOneBool =
        typeof(Z3Methods).GetMethod(nameof(Z3Methods.ExactlyOne))!;

    public MethodCallExpression Rewrite(MethodCallExpression call)
    {
        var args = call.Arguments;
        switch (call.Method.Name)
        {
            case nameof(PredicatsMetier.TousDifferents):
                // TousDifferents(a, b, c)  =>  Z3Methods.Distinct(a, b, c)   (params int[])
                return Expression.Call(DistinctInt, Expression.NewArrayInit(typeof(int), args));
            case nameof(PredicatsMetier.UneSeuleVraie):
                // UneSeuleVraie(p, q)  =>  Z3Methods.ExactlyOne(p, q)        (params bool[])
                return Expression.Call(ExactlyOneBool, Expression.NewArrayInit(typeof(bool), args));
            default:
                throw new InvalidOperationException($"Predicat metier inconnu : {call.Method.Name}");
        }
    }
}

// Temoin negatif 1 : renvoie l'appel recu -> rejet anti-boucle de la bibliotheque.
public class RewriterIdentite : ITheoremPredicateRewriter
{
    public MethodCallExpression Rewrite(MethodCallExpression call) => call;
}

// Temoin negatif 2 : deplie vers une methode ordinaire, que le traducteur ne connait pas.
public class DeplieurNaif : ITheoremPredicateRewriter
{
    public MethodCallExpression Rewrite(MethodCallExpression call)
    {
        var cible = typeof(FormesNonTraduisibles).GetMethod(nameof(FormesNonTraduisibles.Congru));
        return Expression.Call(cible!, call.Arguments[0], Expression.Constant(2), Expression.Constant(0));
    }
}

public static class FormesNonTraduisibles
{
    public static bool Congru(int valeur, int modulo, int reste) => valeur % modulo == reste;
}

// Le vocabulaire metier, annote : c'est l'appel de CES methodes qui declenche la reecriture.
public static class PredicatsMetier
{
    [TheoremPredicateRewriter(typeof(DeplieurMetier))]
    public static bool TousDifferents(int a, int b, int c) => a != b && b != c && a != c;

    [TheoremPredicateRewriter(typeof(DeplieurMetier))]
    public static bool UneSeuleVraie(bool p, bool q) => p ^ q;

    [TheoremPredicateRewriter(typeof(RewriterIdentite))]
    public static bool Boucle(int x) => true;

    [TheoremPredicateRewriter(typeof(DeplieurNaif))]
    public static bool EstPair(int x) => x % 2 == 0;
}

Console.WriteLine("Definitions pretes : DeplieurMetier (2 predicats), RewriterIdentite, DeplieurNaif, PredicatsMetier");


Definitions pretes : DeplieurMetier (2 predicats), RewriterIdentite, DeplieurNaif, PredicatsMetier



warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, 

## 3. Ce que la réécriture change au verdict

Même théorème que la section 1 — mêmes contraintes écrites, un seul changement : la méthode porte
l'attribut. Si le rewriter est bien celui décrit, la résolution aboutit et le témoin vérifie la
propriété **en C#**, hors de Z3.


In [4]:
// PREUVE 1 : le predicat metier est traduit, le theoreme est resolu.
using (var ctx = new Z3Context())
{
    var theorem = from t in ctx.NewTheorem<Triplet>()
                  where PredicatsMetier.TousDifferents(t.A, t.B, t.C)
                  where t.A > 0 && t.B > 0 && t.C > 0
                  where t.A + t.B + t.C == 6
                  select t;
    var sol = theorem.Solve();
    Console.WriteLine($"« TousDifferents(A,B,C), A,B,C>0, A+B+C=6 » -> {(sol == null ? "UNSAT" : $"(A,B,C) = ({sol.A},{sol.B},{sol.C})")}");
    if (sol != null)
    {
        bool somme = sol.A + sol.B + sol.C == 6;
        bool positifs = sol.A > 0 && sol.B > 0 && sol.C > 0;
        bool distincts = sol.A != sol.B && sol.B != sol.C && sol.A != sol.C;
        Console.WriteLine($"Temoin verifie hors Z3 : somme={somme}, positifs={positifs}, distincts={distincts}");
        Console.WriteLine($"Verdict : {(somme && positifs && distincts ? "solution valide" : "solution INVALIDE")}");
    }
}


« TousDifferents(A,B,C), A,B,C>0, A+B+C=6 » -> (A,B,C) = (3,1,2)


Temoin verifie hors Z3 : somme=True, positifs=True, distincts=True


Verdict : solution valide



warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, 

In [5]:
// PREUVE 2 : second predicat, sur un environnement booleen, avec une reecriture differente.
public class PaireBooleenne
{
    public bool P { get; set; }
    public bool Q { get; set; }
}

using (var ctx = new Z3Context())
{
    var theorem = from t in ctx.NewTheorem<PaireBooleenne>()
                  where PredicatsMetier.UneSeuleVraie(t.P, t.Q)
                  where !t.P
                  select t;
    var sol = theorem.Solve();
    Console.WriteLine($"« UneSeuleVraie(P,Q), !P » -> {(sol == null ? "UNSAT" : $"(P,Q) = ({sol.P},{sol.Q})")}");
    Console.WriteLine("Attendu : P = False, Q = True (exactement un des deux)");
    Console.WriteLine("Temoin verifie hors Z3 : " + (sol != null && sol.P == false && sol.Q == true));
}


« UneSeuleVraie(P,Q), !P » -> (P,Q) = (False,True)


Attendu : P = False, Q = True (exactement un des deux)


Temoin verifie hors Z3 : True



warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime



## 4. Les deux gardes, mesurées

Un mécanisme d'extension sans garde-fou est une invitation à la boucle infinie. Le fork en pose
deux, que cette section déclenche exprès : le rewriter **identité** (qui ne réécrit rien) et le
rewriter **naïf** (qui déplie vers une méthode ordinaire, non traduisible).


In [6]:
// TEMOIN NEGATIF 1 : rewriter identite -> rejet anti-boucle.
using (var ctx = new Z3Context())
{
    try
    {
        var theorem = from t in ctx.NewTheorem<Triplet>()
                      where PredicatsMetier.Boucle(t.A)
                      select t;
        Console.WriteLine($"INATTENDU : resolution aboutie -> {theorem.Solve()}");
    }
    catch (InvalidOperationException ex)
    {
        Console.WriteLine("REJET attendu (InvalidOperationException) :");
        Console.WriteLine("  " + ex.Message);
        bool gardeNommee = ex.Message.Contains("did not perform any rewrite");
        Console.WriteLine("Mesure : le message nomme la non-reecriture = " + gardeNommee);
        if (!gardeNommee)
            throw new InvalidOperationException(
                "Temoin 1 : la garde anti-boucle n'est pas nommee dans le message (attendu : did not perform any rewrite).");
    }
}


REJET attendu (InvalidOperationException) :


  The expression tree rewriter of type RewriterIdentite did not perform any rewrite. Aborting compilation to avoid infinite looping.


Mesure : le message nomme la non-reecriture = True



warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime



In [7]:
// TEMOIN NEGATIF 2 : deplier vers une methode ordinaire ne suffit pas.
// Le traducteur ne fait pas d'inlining de corps : la cible doit etre une forme traduisible.
using (var ctx = new Z3Context())
{
    try
    {
        var theorem = from t in ctx.NewTheorem<Triplet>()
                      where PredicatsMetier.EstPair(t.A)
                      select t;
        Console.WriteLine($"INATTENDU : resolution aboutie -> {theorem.Solve()}");
    }
    catch (NotSupportedException ex)
    {
        Console.WriteLine("REFUS attendu (NotSupportedException) :");
        Console.WriteLine("  " + ex.Message);
        bool refusSurCible = ex.Message.StartsWith("Unknown method call");
        Console.WriteLine("Mesure : le refus porte sur l'appel CIBLE, pas sur le predicat d'origine = " + refusSurCible);
        if (!refusSurCible)
            throw new InvalidOperationException(
                "Temoin 2 : le refus devait porter sur l'appel CIBLE (Unknown method call), message observe : " + ex.Message);
    }
}


REFUS attendu (NotSupportedException) :


  Unknown method call:Boolean Congru(Int32, Int32, Int32)


Mesure : le refus porte sur l'appel CIBLE, pas sur le predicat d'origine = True



warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime



### Lecture du résultat

Les deux refus sont complémentaires, et c'est le second qui porte la leçon de conception :

- **identité** → la bibliothèque refuse une réécriture qui n'avance pas (terminaison garantie) ;
- **cible non traduisible** → la réécriture a bien eu lieu, mais l'expression rendue est re-visitée
  telle quelle et rejetée, parce que le traducteur ne connaît pas `Congru`.

Autrement dit : le rewriter **ne fait pas de macro-expansion de C#**. Il traduit un vocabulaire
métier vers la **grammaire du traducteur**, pas vers du code arbitraire. C'est une contrainte à
connaître avant d'en écrire un — et elle explique pourquoi la cible utile est presque toujours un
opérateur de `Z3Methods`.

La correspondance est donc déclarative : `TousDifferents(a,b,c)` **est** `Distinct(a,b,c)`,
`UneSeuleVraie(p,q)` **est** `ExactlyOne(p,q)`. Le carnet garde le mot métier, le solveur reçoit
l'opérateur qu'il sait propager.


## 5. Le rewriter global : réécrire l'ensemble des contraintes

Le second mécanisme change d'échelle. Lu dans le fork (`Theorem.GetConstraintsToAssert`,
`Theorem.Explain`) :

- on annote le **type d'environnement** avec `[TheoremGlobalRewriter(typeof(...))]` ;
- à chaque résolution, la bibliothèque instancie le rewriter et lui passe **toutes** les contraintes
  accumulées, dans l'ordre de leur déclaration ;
- le rewriter rend les contraintes **à asserter à la place** : il peut en ajouter, en supprimer,
  en remplacer.

Conséquence architecturale, écrite dans le fork : `GetConstraintsToAssert` est **partagé par le
chemin de résolution et par `Explain`**. Le diagnostic porte donc sur les contraintes
**réécrites** — pas sur celles que vous avez écrites. C'est la cohérence voulue : rapporter un
conflit sur des contraintes que le solveur n'a jamais vues serait un mensonge de diagnostic.

Premier effet, le plus net : une contrainte **supprimée** rend satisfiable un système qui ne
l'était pas.


In [8]:
// Environnement SANS rewriter : systeme contradictoire (X > 5 et X < 3).
public class SansRewriter
{
    public int X { get; set; }
}

// Rewriter global : ecarte les contraintes dont la forme textuelle contient "t.X < 3".
// Filtre volontairement simple et documente : c'est une demonstration de mecanisme,
// pas une heuristique de production.
public class RelaxBorneHaute : ITheoremGlobalRewriter
{
    public IEnumerable<LambdaExpression> Rewrite(IEnumerable<LambdaExpression> constraints)
    {
        var recues = constraints.ToList();
        Console.WriteLine($"  [rewriter global] {recues.Count} contrainte(s) recue(s)");
        var conservees = recues.Where(c => !c.Body.ToString().Contains("t.X < 3")).ToList();
        Console.WriteLine($"  [rewriter global] {recues.Count - conservees.Count} supprimee(s), {conservees.Count} conservee(s)");
        return conservees;
    }
}

[TheoremGlobalRewriter(typeof(RelaxBorneHaute))]
public class AvecRelaxation
{
    public int X { get; set; }
}

using (var ctx = new Z3Context())
{
    var dur = from t in ctx.NewTheorem<SansRewriter>()
              where t.X > 5
              where t.X < 3
              select t;
    var solDur = dur.Solve();
    Console.WriteLine($"SANS rewriter : « X>5, X<3 » -> {(solDur == null ? "UNSAT (null)" : $"X = {solDur.X}")}");

    Console.WriteLine("AVEC rewriter global sur les memes contraintes ecrites :");
    var relax = from t in ctx.NewTheorem<AvecRelaxation>()
                where t.X > 5
                where t.X < 3
                select t;
    var solRelax = relax.Solve();
    Console.WriteLine($"-> {(solRelax == null ? "UNSAT (null)" : $"X = {solRelax.X}")}");
    bool bascule = solDur == null && solRelax != null && solRelax.X > 5;
    Console.WriteLine("Mesure : le verdict a bascule UNSAT -> SAT = " + bascule);
    if (!bascule)
        throw new InvalidOperationException(
            "Rewriter global : le verdict devait basculer UNSAT -> SAT (X > 5) -- verdict inchange.");
}


SANS rewriter : « X>5, X<3 » -> UNSAT (null)


AVEC rewriter global sur les memes contraintes ecrites :


  [rewriter global] 2 contrainte(s) recue(s)


  [rewriter global] 1 supprimee(s), 1 conservee(s)


-> X = 6


Mesure : le verdict a bascule UNSAT -> SAT = True



warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, 

### Lecture du résultat

Le rewriter global est un **transformateur de spécification** : il s'exécute une fois par
résolution, sur l'ensemble des contraintes, et ce qu'il rend *est* le problème posé au solveur.
Les usages canoniques : injecter des contraintes de domaine implicites (toutes les quantités sont
positives), dédupliquer des contraintes générées, appliquer une politique de relaxation,
journaliser la spécification effective.

Le prix est symétrique et doit être dit : **ce que vous écrivez n'est plus nécessairement ce que
le solveur résout**. D'où la section suivante — un rewriter global doit être documenté là où il
s'emploie, et le diagnostic doit rester lisible.


## 6. La cohérence `Explain`, montrée et non affirmée

`Explain()` rend un `SolveStatus` et, sur `UNSAT`, le **cœur insatisfiable** : le sous-ensemble de
contraintes qui participent réellement au conflit, chacune référencée par son index et son texte.

Le rewriter suivant **ajoute** au système une contrainte de domaine que l'appelant n'a jamais
écrite. Si le chemin était partagé, `Explain` doit voir cette contrainte injectée dans le cœur ;
s'il ne l'était pas, le cœur ne pourrait nommer que le `t.X < 0` d'origine, et le diagnostic
serait faux. C'est exactement le test que nous exécutons.


In [9]:
// Rewriter global qui AJOUTE une contrainte de domaine : "les quantites sont positives".
public class InjecteDomainePositif : ITheoremGlobalRewriter
{
    public IEnumerable<LambdaExpression> Rewrite(IEnumerable<LambdaExpression> constraints)
    {
        var liste = constraints.ToList();
        Expression<Func<AvecDomaine, bool>> domaine = t => t.X >= 0;
        Console.WriteLine($"  [injection] {liste.Count} contrainte(s) ecrite(s) + 1 contrainte de domaine (X >= 0)");
        return liste.Concat(new LambdaExpression[] { domaine });
    }
}

[TheoremGlobalRewriter(typeof(InjecteDomainePositif))]
public class AvecDomaine
{
    public int X { get; set; }
}

using (var ctx = new Z3Context())
{
    var theorem = from t in ctx.NewTheorem<AvecDomaine>()
                  where t.X < 0
                  select t;

    var sol = theorem.Solve();
    Console.WriteLine($"Solve()   : « X<0 » avec domaine injecte -> {(sol == null ? "UNSAT (null)" : $"X = {sol.X}")}");

    var explication = theorem.Explain();
    Console.WriteLine($"Explain() : statut = {explication.Status}, coeur de {explication.UnsatCore.Count} contrainte(s)");
    foreach (var c in explication.UnsatCore)
    {
        Console.WriteLine($"  {c}");
    }
    bool domaineCite = explication.UnsatCore.Any(c => c.Expression.Contains("X >= 0"));
    Console.WriteLine("Mesure : le coeur cite la contrainte INJECTEE par le rewriter = " + domaineCite);
    if (!domaineCite)
        throw new InvalidOperationException(
            "Explain() : le coeur devait citer la contrainte injectee (X >= 0) -- absente du coeur.");
}


  [injection] 1 contrainte(s) ecrite(s) + 1 contrainte de domaine (X >= 0)


Solve()   : « X<0 » avec domaine injecte -> UNSAT (null)


  [injection] 1 contrainte(s) ecrite(s) + 1 contrainte de domaine (X >= 0)


Explain() : statut = Unsatisfiable, coeur de 2 contrainte(s)


  #0: (t.X < 0)


  #1: (t.X >= 0)


Mesure : le coeur cite la contrainte INJECTEE par le rewriter = True



warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime



### Lecture du résultat

Le cœur insatisfiable nomme **deux** contraintes alors que l'appelant n'en a écrit qu'une : la
sienne, et celle que le rewriter a ajoutée. C'est la preuve visible que `Solve` et `Explain`
partagent le même chemin de contraintes — et la raison pour laquelle un rewriter global engage
aussi le diagnostic.

La règle d'usage qui en découle : un rewriter global se documente à l'endroit où il s'applique, et
son effet doit rester inspectable (ici, il s'annonce lui-même sur la sortie). Une spécification qui
change en silence rendrait le cœur insatisfiable ininterprétable pour la personne qui lit le
rapport d'échec.


## 7. Les deux points d'extension, côte à côte

| | **Rewriter de prédicat** | **Rewriter global** |
|---|---|---|
| Attribut | `[TheoremPredicateRewriter]` sur une **méthode** | `[TheoremGlobalRewriter]` sur le **type d'environnement** |
| Déclencheur | chaque **appel** de cette méthode, pendant la traduction | une fois par **résolution** (et par `Explain`) |
| Entrée | le `MethodCallExpression` de l'appel | `IEnumerable<LambdaExpression>` — toutes les contraintes |
| Sortie | un appel à traduire **à la place** (doit différer de l'entrée) | les contraintes **à asserter** |
| Exigence | constructeur public sans paramètre | constructeur public sans paramètre |
| Garde mesurée | identité rejetée (anti-boucle) **et** cible non traduisible rejetée | aucune garde : la liberté inclut de vider le système |
| Effet observé ici | un prédicat refusé devient résoluble | un `UNSAT` devient `SAT` ; un `SAT` devient `UNSAT` |
| Cas d'usage | vocabulaire métier : `TousDifferents`, `UneSeuleVraie`, `EstPair`… | domaines implicites, déduplication, relaxation, instrumentation |
| Risque | cible hors grammaire du traducteur | l'énoncé résolu n'est plus celui écrit — documenter |

Les deux mécanismes sont **additifs** : aucune modification du fork n'est nécessaire, et le carnet
reste sur la build `.deploy/` partagée par toute la série.


## 8. Exercices

Stubs conformes C.1 : le carnet s'exécute de bout en bout même non complétés.


In [10]:
// EXERCICE 1 — Un troisieme predicat metier.
// Ecrivez PredicatsMetier.AuPlusUneVraie(bool a, bool b, bool c), annotez-la avec DeplieurMetier
// (ou un rewriter dedie), et faites-la correspondre a Z3Methods.AtMostOne(a, b, c).
// Resolvez ensuite : AuPlusUneVraie(P,Q,R) avec P et Q vrais -> attendu R = False.
// Indice : deux moities obligatoires. (1) La cible attend un params bool[] ;
// Expression.NewArrayInit construit ce tableau. (2) Si vous reutilisez DeplieurMetier,
// ajoutez AUSSI le case AuPlusUneVraie dans le switch de DeplieurMetier.Rewrite : son
// default leve "Predicat metier inconnu" pour tout appel non liste -- l'attribut seul
// ne suffit pas (cf. temoin negatif en debut de carnet). Un rewriter dedie evite cette
// deuxieme moitie.
// TODO etudiant
Console.WriteLine("Exercice 1 a completer : AuPlusUneVraie -> AtMostOne (stub C.1, aucune erreur volontaire)");


Exercice 1 a completer : AuPlusUneVraie -> AtMostOne (stub C.1, aucune erreur volontaire)


In [11]:
// EXERCICE 2 — Rewriter global de deduplication.
// Construisez un environnement dont les contraintes ecrites contiennent DEUX fois "t.X > 5",
// plus "t.X < 8". Ecrivez un rewriter global qui deduplique par c.Body.ToString(), affichez
// combien il en retire, et verifiez que la solution obtenue est inchangee.
// Question : pourquoi dedupliquer par ToString() est-il fragile des que le rewriter de predicat
// entre en jeu ?
// TODO etudiant
Console.WriteLine("Exercice 2 a completer : deduplication globale (stub C.1)");


Exercice 2 a completer : deduplication globale (stub C.1)


In [12]:
// EXERCICE 3 — Un rewriter global qui rend le systeme plus CONTRAIGNANT.
// Sur un environnement ou l'on ecrit seulement "t.X > 0", injectez "X != 7" et montrez que la
// solution evite 7. Puis rendez un systeme contradictoire par injection ("X < 0" plus injection
// "X >= 0") et verifiez que le coeur rendu par Explain() cite la contrainte injectee.
// Redigez en une phrase la regle d'usage : pourquoi un rewriter global doit-il etre documente
// dans le carnet qui l'emploie ?
// TODO etudiant
Console.WriteLine("Exercice 3 a completer : injection contraignante + lecture du coeur Explain (stub C.1)");


Exercice 3 a completer : injection contraignante + lecture du coeur Explain (stub C.1)


## 9. Conclusion

Le DSL du fork n'est pas une boîte fermée : deux points d'extension permettent de l'adapter au
vocabulaire d'un domaine **sans toucher à la bibliothèque**.

- Le **rewriter de prédicat** apprend au traducteur un mot nouveau. Sa cible n'est pas du C#
  arbitraire : c'est une forme que la grammaire du traducteur couvre déjà — la démonstration
  négative de la section 4 le mesure, et c'est la contrainte à connaître avant d'en écrire un.
- Le **rewriter global** transforme la spécification entière avant assertion. Il change
  réellement le verdict (mesuré : `UNSAT` → `SAT` par suppression, `SAT` → `UNSAT` par injection),
  et `Explain` rapporte les contraintes **réécrites**, cœur insatisfiable à l'appui.

Le critère de bon usage est le même que pour toute extension : la réécriture doit préserver
l'intention de l'énoncé, et rester **visible** pour qui relit le carnet.
